# Clase 10 - Capa BRONZE - Ingesta de datos crudos

Primer paso de la arquitectura Medallion: leer los 4 CSVs desde Azure Blob Storage y persistirlos en formato Delta **sin transformaciones**.

- Storage Account: `stdiploventas` (West US 2)
- Container: `medallion`
- Archivos de entrada: `Clientes.csv`, `Productos.csv`, `tiendas.csv`, `Ventas_diarias.csv`
- Destino Bronze (Volume Unity Catalog reutilizado de Clase 8): `/Volumes/dbx_diplo_ricardo/default/clase8/clase10/bronze/...`
- Catalogo / esquema para tablas registradas: `dbx_diplo_ricardo.bronze_clase10`

**Nota importante:** reutilizamos el Volume `clase8` que ya existe en tu workspace. No creamos un Volume nuevo porque el comando `CREATE VOLUME` falla con `Missing cloud file system scheme` cuando el catalogo no tiene managed storage location configurada.

## Paso 0 - Definir las rutas

Apuntamos al Volume `clase8` (ya existente) y armamos adentro una subcarpeta `clase10/` para todas las tablas Medallion.

In [ ]:
BASE_VOL = "/Volumes/dbx_diplo_ricardo/default/clase8/clase10"

# Aseguramos las subcarpetas Bronze (Spark las crea solas al escribir, pero las inicializamos por claridad)
dbutils.fs.mkdirs(f"{BASE_VOL}/bronze")
print("Destino Bronze:", f"{BASE_VOL}/bronze")
dbutils.fs.ls(BASE_VOL)

## Paso 1 - Configurar acceso al Storage Account

Pegamos la Access Key1 de `stdiploventas` (Portal -> Storage Account -> Security + networking -> Access keys -> key1 -> Show -> Copy). Debe terminar en `==` y tener 88 caracteres.

In [ ]:
storage_account = "stdiploventas"
container       = "medallion"
account_key     = "TU_KEY_AQUI"  # Pegar Access Key1 de stdiploventas (88 chars, termina en ==)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    account_key
)

BASE_BLOB = f"wasbs://{container}@{storage_account}.blob.core.windows.net"
print("Storage configurado:", BASE_BLOB)

# Listamos el container para verificar que la key funciona
dbutils.fs.ls(BASE_BLOB)

## Paso 2 - Leer los 4 CSVs (todo como string) y escribir Delta en Bronze

En Bronze NO transformamos: `header=True` y `inferSchema=False` (default) para que todos los campos queden como string. Asi minimizamos el riesgo de descartar datos por tipos invalidos.

In [ ]:
from pyspark.sql import SparkSession

# En Databricks ya existe `spark`; lo dejamos por claridad
spark = SparkSession.builder.appName("BronzeIngestaClase10").getOrCreate()

# --- VENTAS DIARIAS ---
ventas_bronze = (spark.read
                      .option("header", True)
                      .csv(f"{BASE_BLOB}/Ventas_diarias.csv"))

(ventas_bronze.write
              .format("delta")
              .mode("overwrite")
              .save(f"{BASE_VOL}/bronze/ventas_diarias"))

print("ventas_diarias filas:", ventas_bronze.count())

# --- CLIENTES ---
clientes_bronze = spark.read.option("header", True).csv(f"{BASE_BLOB}/Clientes.csv")
(clientes_bronze.write.format("delta").mode("overwrite")
                .save(f"{BASE_VOL}/bronze/clientes"))
print("clientes filas:", clientes_bronze.count())

# --- PRODUCTOS ---
productos_bronze = spark.read.option("header", True).csv(f"{BASE_BLOB}/Productos.csv")
(productos_bronze.write.format("delta").mode("overwrite")
                 .save(f"{BASE_VOL}/bronze/productos"))
print("productos filas:", productos_bronze.count())

# --- TIENDAS ---
tiendas_bronze = spark.read.option("header", True).csv(f"{BASE_BLOB}/tiendas.csv")
(tiendas_bronze.write.format("delta").mode("overwrite")
               .save(f"{BASE_VOL}/bronze/tiendas"))
print("tiendas filas:", tiendas_bronze.count())

## Paso 3 - Registrar las tablas Bronze en Unity Catalog

Con esto las podemos consultar como `dbx_diplo_ricardo.bronze_clase10.bronze_<tabla>` desde SQL.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_diplo_ricardo.bronze_clase10")

for tabla in ["ventas_diarias", "clientes", "productos", "tiendas"]:
    spark.sql(f"DROP TABLE IF EXISTS dbx_diplo_ricardo.bronze_clase10.bronze_{tabla}")
    spark.sql(f"""
        CREATE TABLE dbx_diplo_ricardo.bronze_clase10.bronze_{tabla}
        USING DELTA
        LOCATION '{BASE_VOL}/bronze/{tabla}'
    """)

spark.sql("SHOW TABLES IN dbx_diplo_ricardo.bronze_clase10").show(truncate=False)

## Paso 4 - Verificacion final

Releemos cada tabla Delta y mostramos schema (deberia ser todo string) + primeras filas.

In [ ]:
for tabla in ["ventas_diarias", "clientes", "productos", "tiendas"]:
    print("=" * 60)
    print("BRONZE -", tabla)
    df = spark.read.format("delta").load(f"{BASE_VOL}/bronze/{tabla}")
    df.printSchema()
    df.show(5, truncate=False)

In [ ]:
%sql
SELECT * FROM dbx_diplo_ricardo.bronze_clase10.bronze_ventas_diarias LIMIT 10;